# 01 — Carga y Exploración de Datos

Este notebook carga el dataset combinado (SCADA + OPC UA), filtra al período de análisis de 1 año, añade variables temporales y genera una copia limpia lista para la inyección de anomalías.

**Salida:** `data/interim/01_datos_cargados.parquet`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 1. Cargar el dataset
Leemos el CSV combinado y filtramos al período de análisis (1 año: 2024-03-06 → 2025-03-07).

In [ ]:
# Fichero: combined (SCADA + OPC UA fusionados) — 1 año completo
# Período: 2024-03-06 → 2025-03-07
# DATASET_PATH viene de config.py y apunta a la ruta correcta independientemente
# de desde qué carpeta se ejecute el notebook
df_original = pd.read_csv(DATASET_PATH)

# Filtrar a un año completo de datos
df_original['Fecha'] = pd.to_datetime(df_original['Fecha'], format='%d/%m/%Y %H:%M:%S')
df_original = df_original[df_original['Fecha'] <= '2025-03-07'].reset_index(drop=True)

print(f'Dataset cargado: {df_original.shape[0]:,} filas × {df_original.shape[1]} columnas')
print(f'Período: {df_original["Fecha"].min()} → {df_original["Fecha"].max()}')
print(df_original.head())

print(f'Shape tras filtro de fecha: {df_original.shape}')

## 2. Verificar datos faltantes
Contamos NaNs por columna para tener una línea base antes de cualquier manipulación.

In [ ]:
if 'df_original' in locals():
    print("\nConteo de datos faltantes (NaN) por columna:")
    missing_values = df_original.isnull().sum()
    print(missing_values[missing_values > 0]) # Muestra solo columnas con datos faltantes
    if missing_values.sum() == 0:
        print("No se encontraron datos faltantes (NaN) explícitos en el dataset.")

## 3. Añadir variables temporales
Extraemos Hora, DiaSemana y Mes del campo Fecha. Ayudan al modelo a detectar anomalías temporales (p.ej. luz nocturna anómala).

In [ ]:
# Convertir la columna de fecha a datetime
df_original['Fecha'] = pd.to_datetime(df_original['Fecha'], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# Extraer características temporales que podrían ser útiles
df_original['Hora'] = df_original['Fecha'].dt.hour
df_original['DiaSemana'] = df_original['Fecha'].dt.dayofweek
df_original['Mes'] = df_original['Fecha'].dt.month

## 4. Exploración rápida del dataset

In [ ]:
if 'df_original' in locals():
    print("\nNombres de columnas originales:")
    print(df_original.columns.tolist())

if 'df_original' in locals():
    num_duplicates = df_original.duplicated().sum()
    if num_duplicates > 0:
        print(f"\nSe encontraron {num_duplicates} filas duplicadas completas.")
    else:
        print("\nNo se encontraron filas duplicadas completas en el dataset.")

## 5. Crear copia de trabajo
Inicializamos `df_trabajo` con columnas de etiquetas. Todo empieza como 'normal'.

In [ ]:
# Crear una copia del DataFrame original para trabajar
df_trabajo = df_original.copy()
print("Copia del DataFrame creada como 'df_trabajo'.")
# Inicializar la columna para la etiqueta de detección de anomalías
# Inicialmente, todas las filas se consideran "normales"
df_trabajo['etiqueta_deteccion'] = 'normal'
print("Columna 'etiqueta_deteccion' inicializada a 'normal'.")
# Inicializar la columna para el tipo específico de anomalía
# Inicialmente, no hay tipo de anomalía asignado (NaN o None)
df_trabajo['etiqueta_tipo_anomalia'] = pd.NA # Usar pd.NA para datos faltantes de pandas

print("Columna 'etiqueta_tipo_anomalia' inicializada a NA.")
# Verificar las nuevas columnas
print("\nPrimeras filas de 'df_trabajo' con las nuevas columnas de etiquetas:")
print(df_trabajo.head())
print("\nConteo de valores en 'etiqueta_deteccion':")
print(df_trabajo['etiqueta_deteccion'].value_counts())
print("\nConteo de valores en 'etiqueta_tipo_anomalia' (deberían ser todos NA/NaN por ahora):")
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)) # dropna=False para contar los NA


## 6. Guardar resultado

In [ ]:
os.makedirs(DATA_INTERIM, exist_ok=True)
df_trabajo.to_parquet(PARQUET_01, index=False)
print(f"Guardado: {PARQUET_01}")
print(f"Shape: {df_trabajo.shape}")
